# 01 — Sinh prompt B0, giữ khung B2 và để trống graph/demo
Chạy local/WSL trong repo GRACE-summary. Dùng frozen raw validation/test, không đọc donor hoặc chạy retrieval/Joern/CodeT5.
Giữ vai trò, miền, câu hỏi, câu dẫn node/edge/demo và Answer của B2. Xóa dữ liệu node/edge, code demo và **nhãn demo**; câu dẫn demo trở thành `The following is an example, and it is .`.
Đây là biến thể `b0_b2_empty_blocks_v1` theo yêu cầu, khác B0 bỏ hẳn khối graph/demo trong PDF. Cắt code target bằng đúng TokenCounter và marker B2, tối đa 3.000 token; tổng 8.192 gồm chat template, dự trữ 64.


In [1]:
from pathlib import Path
import sys, subprocess
# Đặt True nếu môi trường chưa có thư viện; không cần GPU để sinh prompt.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.1','numpy','sentencepiece','pyyaml'],check=True)
# Có thể điền đường dẫn folder 02_Arm_B0_Source_Only nếu kernel mở ngoài repo.
ARM_FOLDER_OVERRIDE = None
candidates = [Path(ARM_FOLDER_OVERRIDE)] if ARM_FOLDER_OVERRIDE else [p/'02_Arm_B0_Source_Only' for p in [Path.cwd(),*Path.cwd().parents]] + [Path.cwd(),Path.cwd().parent]
ARM_ROOT = next((p.resolve() for p in candidates if (p/'prepare_b0.py').is_file()),None)
if ARM_ROOT is None: raise FileNotFoundError('Mở notebook trong folder arm hoặc đặt ARM_FOLDER_OVERRIDE')
sys.path.insert(0,str(ARM_ROOT))
from prepare_b0 import prepare, render_b0, TOKENIZER, REVISION, BASELINE
FREEZE_ROOT = BASELINE/'output/research_freeze_20261101'
OUT_ROOT = ARM_ROOT/'output/bundles'
RUNS = [('devign','validation'),('reveal','validation'),('devign','test'),('reveal','test')]
print('Arm:',ARM_ROOT)
print('Frozen input:',FREEZE_ROOT)
print('Output:',OUT_ROOT)


Arm: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only
Frozen input: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/research_freeze_20261101
Output: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles


In [2]:
import hashlib,json
import yaml
protocol=yaml.safe_load((BASELINE/'diagnostic_kit/configs/protocol.yaml').read_text(encoding='utf-8'))
actual=hashlib.sha256((FREEZE_ROOT/'freeze_manifest.json').read_bytes()).hexdigest()
assert actual==protocol['dedup']['manifest_sha256'],'Frozen manifest mismatch'
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(TOKENIZER,revision=REVISION,trust_remote_code=False)
assert tokenizer.chat_template,'Tokenizer must include the model chat template'
print('Tokenizer pinned:',TOKENIZER,REVISION)


/mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer pinned: Qwen/Qwen3-Coder-30B-A3B-Instruct b2cff646eb4bb1d68355c01b18ae02e7cf42d120


In [3]:
# Xem thử khung trước khi xuất tất cả. Nhãn target không được truyền vào renderer.
from grace_diag.token_budget import TokenCounter
counter=TokenCounter(None); counter.tok=tokenizer; counter.approx=False
source=FREEZE_ROOT/RUNS[0][0]/f'{RUNS[0][1]}_raw.jsonl'
with source.open(encoding='utf-8') as f: first=json.loads(next(f))
preview,metadata=render_b0(first['func'],counter)
print(preview)
print(json.dumps(metadata,ensure_ascii=False,indent=2))


You are now an excellent programmer. You are conducting a function vulnerability detection task for C/C++ language.

static void filter_mirror_setup(NetFilterState *nf, Error **errp)
{
    MirrorState *s = FILTER_MIRROR(nf);
    Chardev *chr;
    chr = qemu_chr_find(s->outdev);
    if (chr == NULL) {
        error_set(errp, ERROR_CLASS_DEVICE_NOT_FOUND,
                  "Device '%s' not found", s->outdev);
    qemu_chr_fe_init(&s->chr_out, chr, errp);

In the above code snippet, check for potential security vulnerabilities and answer with a single digit: 1 if Vulnerable, 0 if Non-vulnerable.

The node information of the code snippet is as follows:

The edge information of the code snippet is as follows:


The following is an example, and it is .


Answer (0 or 1):
{
  "context_limit_tokens": 8192,
  "prompt_tokens": 197,
  "target_tokens_used": 97,
  "target_truncated": false,
  "graph_tokens_used": 0,
  "demo_tokens_used": 0,
  "graph_nodes_used": 0,
  "graph_edges_used": 0,
  "graph

In [4]:
bundles,archive=prepare(FREEZE_ROOT,OUT_ROOT,RUNS,tokenizer)
print('Upload lên Kaggle Dataset (account bất kỳ):',archive)
for path in bundles: print(path)


devign validation 2732 prompts; 60 truncated; /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/devign/validation/devign_validation_b0_kaggle.zip
reveal validation 2273 prompts; 34 truncated; /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/reveal/validation/reveal_validation_b0_kaggle.zip
devign test 2732 prompts; 47 truncated; /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/devign/test/devign_test_b0_kaggle.zip
reveal test 2274 prompts; 47 truncated; /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/reveal/test/reveal_test_b0_kaggle.zip
Upload lên Kaggle Dataset (account bất kỳ): /mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/grace_b0_inputs.zip
/mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/devign/validation/devign_validation_b0_kaggle.zip
/mnt/d/JUNIOR/LabThayTho/GRACE-summary/02_Arm_B0_Source_Only/output/bundles/reveal

Upload `output/bundles/grace_b0_inputs.zip` vào một Kaggle Dataset. Bạn có thể giữ ZIP nguyên hoặc để Kaggle giải nén; notebook 02 hỗ trợ cả hai và không phụ thuộc tên Dataset/username. Thêm model Q4_K_M làm Input nếu đã có; notebook có thể tải Hub khi chưa có model. Notebook này chỉ tạo bundle, không gọi LLM.
